# Kaggle runner — E2: DPHGNN component ablation

Clones the `dphgnn_gf` branch, sets up the env (same pattern as the official `run_evaluation.ipynb` launch and as `lifting_confounding_study/kaggle_run_lifting_ablation.ipynb`), then runs the E2 ablation experiment: `run_e2.py` (training) followed by `component_ablation.ipynb` (analysis + figures).

See `2026_tdl_challenge/extra_analysis_oversmooth_operators/model_ablation/README.md` for the experiment design (six ablation arms, H1/H2/H3).

**Run the cells top to bottom. Do not skip the smoke test** — it also runs `run_e2.py`'s `preflight_check()`, which verifies each `+model.backbone.*` ablation override actually takes effect on *this* Kaggle GPU before you commit the real sweep budget.

In [ ]:
import os, shutil, subprocess
os.chdir("/kaggle/working")
print("cwd =", os.getcwd())

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Pas de GPU attaché — vérifie Accelerator dans le panneau de droite.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 1. Clone the branch and set up the environment

In [ ]:
%%bash
cd /kaggle/working          # <<< INDISPENSABLE : recale le sous-shell
set -eo pipefail

DEST=/kaggle/working/topobench
BRANCH=e2_e3_experiments
REPO=https://github.com/yeli-falk/topobench.git
CUDA_VARIANT=cu118

export UV_CACHE_DIR=/tmp/uv-cache
export UV_LINK_MODE=copy

if [ -d "$DEST/.git" ]; then
  git -C "$DEST" fetch origin "$BRANCH"
  git -C "$DEST" checkout "$BRANCH"
  git -C "$DEST" reset --hard "origin/$BRANCH"
else
  rm -rf "$DEST"
  git clone --branch "$BRANCH" "$REPO" "$DEST"
fi

cd "$DEST"

pip install -q -U uv
export PATH="$HOME/.local/bin:$PATH"
uv --version

source uv_env_setup.sh "$CUDA_VARIANT"

# psutil: hard dependency of the orchestrator (RAM diagnostics between
# jobs), not in pyproject.toml. statsmodels is optional (nicer Phase-2
# stats table; the notebook falls back to a manual computation if absent).
uv pip install -q ipykernel nbconvert jupyter-client psutil statsmodels
python -m ipykernel install --user --name=topobench --display-name "Python 3.11 (topobench)"

python - <<'PY'
import torch
print(f"Torch {torch.__version__} | CUDA dispo: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}")
PY

# uv_env_setup.sh rewrites pyproject.toml's PyG find-links/index in
# place for the chosen CUDA variant -- expected to show as modified,
# do not commit it back.
git status --porcelain pyproject.toml


## 2. Preflight + smoke test on the real Kaggle GPU (recommended, ~5 min)

`run_e2.py` runs `preflight_check()` automatically before any sweep (unless `--skip-preflight`): it composes each of the 6 arms' Hydra overrides and asserts the resulting model actually differs from `A0` on a toy forward pass — catching a silently-ignored ablation flag before it wastes the real GPU budget.

The smoke test itself uses tiny synthetic data + 2 epochs across all 6 arms x 2 cells. Nothing is written to `e2_ablation_results.json` by a smoke test. If this cell errors, stop and fix it before running Phase 1 below — see README.md, "Local testing".

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/model_ablation
../../../.venv/bin/python -u run_e2.py --smoke-test


## 3. Phase 1 — the real sweep (12 runs, seed 42)

6 arms (`A0`..`A5`) x 2 cells (`h_lo__d_lo__pl_lo`, `h_hi__d_lo__pl_lo`) x seed 42. Resumable: safe to stop and re-run this cell, already-completed `(arm, cell, seed)` triples are skipped. Each job runs in its own subprocess — if one dies (OOM, driver hiccup), it's recorded as `status: "failed"` in `e2_ablation_results.json` and the sweep continues with the next job instead of stopping.

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/model_ablation
../../../.venv/bin/python -u run_e2.py


## 4. Phase 2 (optional — only run once Phase 1 above is fully green)

Adds seeds 43 and 44 (36 runs total). Enables the bootstrap-CI statistics in the analysis notebook (see README.md, "Statistics"). Skip this cell if you're short on GPU budget — Phase 1 alone is still reportable, just qualitative (n=1 seed).

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/model_ablation
../../../.venv/bin/python -u run_e2.py --phase2


## 5. Analysis — populate the figures and the finding

Only reads `e2_ablation_results.json` and plots — never trains. Writes `figures/fig3_ablation_delta.png` and `figures/fig4_component_by_regime.png` in place.

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/model_ablation
../../../.venv/bin/python -m jupyter nbconvert \
    --to notebook --execute --inplace \
    --ExecutePreprocessor.kernel_name=topobench \
    --ExecutePreprocessor.timeout=600 \
    component_ablation.ipynb


## 6. Sanity check the outputs

In [ ]:
import json
from pathlib import Path

exp_dir = Path(
    "/kaggle/working/topobench/2026_tdl_challenge/"
    "extra_analysis_oversmooth_operators/model_ablation"
)

results_path = exp_dir / "e2_ablation_results.json"
if results_path.exists():
    records = json.loads(results_path.read_text())
    ok = sum(1 for r in records if r.get("status") == "ok")
    failed = [r for r in records if r.get("status") != "ok"]
    print(f"{results_path}: {len(records)} record(s), {ok} ok, {len(failed)} failed")
    for r in failed:
        print(f"  FAILED {r.get('arm')}/{r.get('cell_key')}/s{r.get('seed')}: {r.get('error')}")
else:
    print(f"{results_path} does not exist yet — run Phase 1 (cell above) first.")

figures_dir = exp_dir / "figures"
figs = sorted(figures_dir.glob("*.png")) if figures_dir.exists() else []
print(f"\nFigures in {figures_dir}:")
for f in figs:
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not figs:
    print("  (none yet — run the analysis cell above)")
